In [ ]:
"""
Korelacja tempa i temperatury toru – F1 2023, wszystkie wyścigi, wszyscy kierowcy
Źródło danych: OpenF1 API (https://openf1.org)
pip install requests pandas numpy matplotlib scipy
"""
import os
import json
import time

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

BASE = "https://api.openf1.org/v1"
CACHE_DIR = "openf1_cache"
os.makedirs(CACHE_DIR, exist_ok=True)


# ---------- Pobieranie danych (z cache i obsługą limitów API) ----------
def get(endpoint, **params):
    key = endpoint + "_" + "_".join(f"{k}-{v}" for k, v in sorted(params.items()))
    path = os.path.join(CACHE_DIR, key + ".json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)

    for attempt in range(8):
        r = requests.get(f"{BASE}/{endpoint}", params=params, timeout=120)
        if r.status_code == 429:  # za dużo zapytań – czekamy
            try:
                wait = float(r.headers.get("Retry-After", ""))
            except ValueError:
                wait = 5 * (attempt + 1)
            print(f"  limit API, czekam {wait:.0f}s...")
            time.sleep(wait)
            continue
        r.raise_for_status()
        data = r.json()
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)
        time.sleep(0.6)  # grzecznie wobec darmowego API
        return data
    raise RuntimeError(f"Nie udało się pobrać {endpoint} {params}")


# ---------- Lista wyścigów 2023 ----------
sessions = pd.DataFrame(get("sessions", year=2023, session_name="Race"))
sessions = sessions.sort_values("date_start").reset_index(drop=True)
print(f"Znaleziono {len(sessions)} wyścigów")

all_laps = []

for _, s in sessions.iterrows():
    sk = s["session_key"]
    race = s.get("circuit_short_name") or s.get("location")
    print(f"Pobieram: {race} (session_key={sk})")

    laps = pd.DataFrame(get("laps", session_key=sk))
    weather = pd.DataFrame(get("weather", session_key=sk))
    drivers = pd.DataFrame(get("drivers", session_key=sk))
    if laps.empty or weather.empty:
        print("  brak danych, pomijam")
        continue

    # --- czyszczenie okrążeń ---
    laps = laps.dropna(subset=["lap_duration", "date_start"])
    laps = laps[laps["lap_number"] > 1]
    laps = laps[~laps["is_pit_out_lap"].fillna(False).astype(bool)]
    laps["date_start"] = pd.to_datetime(laps["date_start"], utc=True, format="ISO8601")

    med = laps["lap_duration"].median()
    laps = laps[laps["lap_duration"] < 1.07 * med].copy()   # bez SC/VSC/wjazdów do boksu
    laps["pace_pct"] = (laps["lap_duration"] / med - 1) * 100  # % wolniej niż mediana

    # --- dopasowanie pogody do okrążenia (najbliższy pomiar) ---
    weather["date"] = pd.to_datetime(weather["date"], utc=True, format="ISO8601")
    weather = weather.sort_values("date")
    laps = pd.merge_asof(
        laps.sort_values("date_start"),
        weather[["date", "track_temperature", "air_temperature", "rainfall"]],
        left_on="date_start", right_on="date",
        direction="nearest", tolerance=pd.Timedelta("3min"),
    )
    laps = laps.dropna(subset=["track_temperature"])
    laps = laps[laps["rainfall"].fillna(0) == 0]  # bez okrążeń w deszczu
    if len(laps) < 50:
        print("  za mało czystych okrążeń, pomijam")
        continue

    # --- usunięcie trendu paliwa (liniowo względem numeru okrążenia) ---
    coef = np.polyfit(laps["lap_number"], laps["pace_pct"], 1)
    laps["pace_resid"] = laps["pace_pct"] - np.polyval(coef, laps["lap_number"])
    laps["temp_dev"] = laps["track_temperature"] - laps["track_temperature"].mean()

    # --- kierowcy ---
    if not drivers.empty:
        laps = laps.merge(
            drivers[["driver_number", "name_acronym"]].drop_duplicates("driver_number"),
            on="driver_number", how="left",
        )
    else:
        laps["name_acronym"] = laps["driver_number"].astype(str)

    laps["race"] = race
    all_laps.append(laps)

df = pd.concat(all_laps, ignore_index=True)
df.to_csv("f1_2023_tempo_temperatura.csv", index=False)
print(f"\nCzyste okrążenia: {len(df)}")

# ---------- Statystyki ----------
r_raw, p_raw = stats.pearsonr(df["track_temperature"], df["pace_pct"])
r_in, p_in = stats.pearsonr(df["temp_dev"], df["pace_resid"])
rho, p_rho = stats.spearmanr(df["temp_dev"], df["pace_resid"])
slope, intercept, *_ = stats.linregress(df["temp_dev"], df["pace_resid"])

print(f"Korelacja surowa (między torami):      r = {r_raw:+.3f} (p={p_raw:.1e})")
print(f"Korelacja w obrębie wyścigu (Pearson): r = {r_in:+.3f} (p={p_in:.1e})")
print(f"Korelacja w obrębie wyścigu (Spearman): ρ = {rho:+.3f} (p={p_rho:.1e})")
print(f"Efekt: +1°C toru ≈ {slope:+.3f}% czasu okrążenia")

per_race = (
    df.groupby("race", sort=False)
      .apply(lambda g: stats.pearsonr(g["temp_dev"], g["pace_resid"])[0]
             if g["temp_dev"].std() > 0 else np.nan)
      .rename("r")
)
per_driver = (
    df.groupby("name_acronym")
      .filter(lambda g: len(g) >= 200)
      .groupby("name_acronym")
      .apply(lambda g: stats.pearsonr(g["temp_dev"], g["pace_resid"])[0])
      .sort_values()
      .rename("r")
)

# ---------- Wykresy ----------
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("F1 2023 – tempo vs temperatura toru (OpenF1)", fontsize=16, weight="bold")

# 1. Surowe dane, kolor = wyścig
ax = axes[0, 0]
races = df["race"].unique()
colors = plt.cm.turbo(np.linspace(0, 1, len(races)))
for race, c in zip(races, colors):
    d = df[df["race"] == race]
    ax.scatter(d["track_temperature"], d["pace_pct"], s=3, alpha=0.3, color=c, label=race)
ax.set_xlabel("Temperatura toru [°C]")
ax.set_ylabel("Tempo względne [% wolniej niż mediana wyścigu]")
ax.set_title(f"Wszystkie okrążenia (między torami) r = {r_raw:+.3f}")
ax.legend(fontsize=6, ncol=2, markerscale=4, loc="upper right")

# 2. W obrębie wyścigu, po usunięciu trendu paliwa
ax = axes[0, 1]
hb = ax.hexbin(df["temp_dev"], df["pace_resid"], gridsize=60, cmap="viridis", mincnt=1, bins="log")
xs = np.linspace(df["temp_dev"].min(), df["temp_dev"].max(), 100)
ax.plot(xs, intercept + slope * xs, color="red", lw=2,
        label=f"r = {r_in:+.3f}, {slope:+.3f}% / °C")
ax.set_xlabel("Odchylenie temp. toru od średniej wyścigu [°C]")
ax.set_ylabel("Tempo po usunięciu trendu paliwa [%]")
ax.set_title("W obrębie wyścigu (właściwa korelacja)")
ax.legend()
fig.colorbar(hb, ax=ax, label="liczba okrążeń (log)")

# 3. Korelacja dla każdego wyścigu
ax = axes[1, 0]
cols = ["tab:red" if v > 0 else "tab:blue" for v in per_race.fillna(0)]
ax.bar(per_race.index, per_race.values, color=cols)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("r Pearsona")
ax.set_title("Korelacja per wyścig (czerwony = cieplej → wolniej)")
ax.tick_params(axis="x", rotation=60, labelsize=8)

# 4. Korelacja dla każdego kierowcy
ax = axes[1, 1]
cols = ["tab:red" if v > 0 else "tab:blue" for v in per_driver]
ax.barh(per_driver.index, per_driver.values, color=cols)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("r Pearsona")
ax.set_title("Korelacja per kierowca (min. 200 okrążeń)")

plt.tight_layout()
plt.savefig("f1_2023_tempo_vs_temperatura.png", dpi=150)
plt.show()